# 02 - Carga de Datos y Mapa Geografico por Comunas**Proyecto:** Indice de Ingresos Operacionales - Cali  **Equipo:** ITT Cali Inteligente - Gobierno de Datos  **Repositorio:** https://github.com/j0rg3c45/Indice_ingresos_operacionales.git## Objetivo1. Clonar/actualizar el repositorio desde GitHub (Colab) o usar carpeta local2. Cargar el archivo Excel del Registro Mercantil 20253. Cargar el GeoJSON de comunas de Cali desde `data/info_geo/`4. Calcular indicadores economicos por comuna5. Generar mapas coropleticos (estatico e interactivo)

## 1. Instalacion de dependencias

In [ ]:
# Descomentar si faltan paquetes (Colab)# !pip install pandas openpyxl geopandas folium mapclassify matplotlib seaborn

## 2. Deteccion de entorno y carga del repositorio

In [ ]:
import osfrom pathlib import Path# Configuracion del repositorioREPO_URL = "https://github.com/j0rg3c45/Indice_ingresos_operacionales.git"REPO_NAME = "Indice_ingresos_operacionales"# Detectar entorno: Colab o LocalEN_COLAB = os.path.exists("/content")if EN_COLAB:    WORK_DIR = Path("/content") / REPO_NAME    if not WORK_DIR.exists():        print(f"Clonando repositorio: {REPO_URL}")        os.system(f"git clone {REPO_URL}")    else:        print("Repositorio ya existe, actualizando...")        os.system(f"cd {WORK_DIR} && git pull")else:    # Local: subir un nivel desde notebooks_py/    WORK_DIR = Path(os.getcwd()).parent    if not (WORK_DIR / "README.md").exists():        WORK_DIR = Path(os.getcwd())DATA_DIR = WORK_DIR / "data"GEO_DIR = DATA_DIR / "info_geo" / "geojson_comunas"OUTPUT_DIR = WORK_DIR / "outputs"OUTPUT_DIR.mkdir(parents=True, exist_ok=True)print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")print(f"Directorio de trabajo: {WORK_DIR}")print(f"Directorio de datos: {DATA_DIR}")print(f"Directorio GeoJSON: {GEO_DIR}")

## 3. Carga del Registro Mercantil 2025

In [ ]:
import pandas as pdimport numpy as np# Buscar el archivo Excel en data/archivos_excel = list(DATA_DIR.glob("*.xlsx"))if archivos_excel:    ARCHIVO_EXCEL = archivos_excel[0]else:    ARCHIVO_EXCEL = DATA_DIR / "Registro mercantil 2025_.xlsx"print(f"Cargando: {ARCHIVO_EXCEL.name}")df = pd.read_excel(ARCHIVO_EXCEL, engine="openpyxl")df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)print(f"Registros: {len(df):,}")print(f"Columnas: {len(df.columns)}")print(f"\nColumnas disponibles:")for i, c in enumerate(df.columns, 1):    print(f"  {i:2d}. {c}")

## 4. Carga del GeoJSON de comunas

In [ ]:
import geopandas as gpdimport zipfile# Buscar GeoJSON en data/info_geo/geojson_path = None# Opcion 1: archivo ya descomprimidogeojson_files = list((DATA_DIR / "info_geo").glob("**/*.geojson"))if geojson_files:    geojson_path = geojson_files[0]else:    # Opcion 2: descomprimir ZIP    zip_files = list((DATA_DIR / "info_geo").glob("*.zip"))    if zip_files:        zip_path = zip_files[0]        extract_dir = DATA_DIR / "info_geo" / zip_path.stem.lower()        extract_dir.mkdir(parents=True, exist_ok=True)        with zipfile.ZipFile(zip_path, "r") as z:            z.extractall(extract_dir)        geojson_files = list(extract_dir.glob("**/*.geojson"))        if geojson_files:            geojson_path = geojson_files[0]if geojson_path:    print(f"GeoJSON cargado: {geojson_path.name}")    gdf_comunas = gpd.read_file(geojson_path)        # Normalizar CRS a WGS84    if gdf_comunas.crs is None:        gdf_comunas = gdf_comunas.set_crs("EPSG:4326")    elif gdf_comunas.crs.to_epsg() != 4326:        gdf_comunas = gdf_comunas.to_crs("EPSG:4326")        print(f"Comunas en GeoJSON: {len(gdf_comunas)}")    print(f"CRS: {gdf_comunas.crs}")    print(f"Columnas: {list(gdf_comunas.columns)}")    print()    print(gdf_comunas[["comuna", "nombre"]].to_string())else:    print("[!] No se encontro GeoJSON. Coloca el archivo en data/info_geo/")    gdf_comunas = None

## 5. Preparacion de indicadores por comuna

In [ ]:
# Identificar columnas clavecol_comuna = [c for c in df.columns if "comuna" in c][0]col_ingresos = [c for c in df.columns if "ingreso" in c][0]col_empleo = [c for c in df.columns if "personal" in c][0]col_tamano = [c for c in df.columns if "tama" in c][0]col_ciiu = [c for c in df.columns if "codigo" in c and "ciiu" in c][0]# Convertir a numericodf[col_ingresos] = pd.to_numeric(df[col_ingresos], errors="coerce")df[col_empleo] = pd.to_numeric(df[col_empleo], errors="coerce")# Filtrar registros con comuna validadf_con_comuna = df[df[col_comuna].notna()].copy()print(f"Registros con comuna: {len(df_con_comuna):,} de {len(df):,}")# Calcular indicadores por comunaindicadores = df_con_comuna.groupby(col_comuna).agg(    total_empresas=(col_comuna, "size"),    ingresos_promedio=(col_ingresos, "mean"),    ingresos_mediana=(col_ingresos, "median"),    ingresos_total=(col_ingresos, "sum"),    empleo_total=(col_empleo, "sum"),    empleo_promedio=(col_empleo, "mean"),).reset_index()# Tasa de microempresasmicro = df_con_comuna[df_con_comuna[col_tamano].str.contains("MICRO", case=False, na=False)]tasa_micro = micro.groupby(col_comuna).size().reset_index(name="n_micro")indicadores = indicadores.merge(tasa_micro, on=col_comuna, how="left")indicadores["pct_micro"] = (indicadores["n_micro"] / indicadores["total_empresas"] * 100).round(1)# Diversidad economica (CIIU distintos)diversidad = df_con_comuna.groupby(col_comuna)[col_ciiu].nunique().reset_index(name="n_ciiu_distintos")indicadores = indicadores.merge(diversidad, on=col_comuna, how="left")# Densidad empresarial (empresas por hectarea - se calcula despues del merge con area)print(f"\nIndicadores calculados para {len(indicadores)} comunas")indicadores.sort_values("total_empresas", ascending=False).head(10)

## 6. Cruce de datos con GeoJSON

In [ ]:
if gdf_comunas is not None:    # El GeoJSON tiene columna 'nombre' con formato "Comuna 6", "Comuna 17"    # El Registro Mercantil tiene formato "Comuna 02", "Comuna 17" (con cero a la izquierda)        # Normalizar: extraer numero y crear clave comun    gdf_comunas["comuna_num"] = gdf_comunas["comuna"].astype(int)    gdf_comunas["comuna_key"] = "Comuna " + gdf_comunas["comuna_num"].astype(str).str.zfill(2)        # Verificar formato en datos del registro    print("Formato en Registro Mercantil:", indicadores[col_comuna].head(5).tolist())    print("Formato en GeoJSON (key):", gdf_comunas["comuna_key"].head(5).tolist())        # Merge    gdf_merged = gdf_comunas.merge(        indicadores,        left_on="comuna_key",        right_on=col_comuna,        how="left"    )        n_match = gdf_merged["total_empresas"].notna().sum()    print(f"\nComunas con datos cruzados: {n_match} de {len(gdf_merged)}")        # Calcular densidad empresarial (empresas por hectarea)    gdf_merged["area_ha"] = gdf_merged["area"] / 10000  # m2 a hectareas    gdf_merged["densidad_empresarial"] = (gdf_merged["total_empresas"] / gdf_merged["area_ha"]).round(2)        if n_match == 0:        print("\n[!] No hubo match. Revisar nombres de comuna.")else:    gdf_merged = None    print("[!] No hay GeoJSON cargado.")

## 7. Mapa coropletico - Total de empresas por comuna

In [ ]:
import matplotlib.pyplot as pltif gdf_merged is not None and gdf_merged["total_empresas"].notna().any():    fig, ax = plt.subplots(1, 1, figsize=(12, 10))        gdf_merged.plot(        column="total_empresas",        cmap="YlOrRd",        linewidth=0.8,        edgecolor="0.3",        legend=True,        legend_kwds={"label": "Total de empresas", "shrink": 0.7},        ax=ax,        missing_kwds={"color": "lightgrey", "label": "Sin datos"}    )        # Etiquetas de comuna    for idx, row in gdf_merged.iterrows():        centroid = row.geometry.centroid        label = str(int(row["comuna_num"])) if pd.notna(row.get("comuna_num")) else ""        ax.annotate(label, xy=(centroid.x, centroid.y), ha="center", fontsize=8, fontweight="bold")        ax.set_title("Densidad Empresarial por Comuna - Cali 2025", fontsize=14, fontweight="bold")    ax.set_axis_off()    plt.tight_layout()        fig.savefig(OUTPUT_DIR / "mapa_total_empresas_comuna.png", dpi=150, bbox_inches="tight")    print("[OK] Guardado: outputs/mapa_total_empresas_comuna.png")    plt.show()else:    print("[!] No se puede generar mapa.")

## 8. Mapa coropletico - Ingresos promedio por comuna

In [ ]:
if gdf_merged is not None and gdf_merged["ingresos_promedio"].notna().any():    fig, ax = plt.subplots(1, 1, figsize=(12, 10))        gdf_merged.plot(        column="ingresos_promedio",        cmap="Blues",        linewidth=0.8,        edgecolor="0.3",        legend=True,        legend_kwds={"label": "Ingresos promedio ($)", "shrink": 0.7},        ax=ax,        missing_kwds={"color": "lightgrey", "label": "Sin datos"}    )        for idx, row in gdf_merged.iterrows():        centroid = row.geometry.centroid        label = str(int(row["comuna_num"])) if pd.notna(row.get("comuna_num")) else ""        ax.annotate(label, xy=(centroid.x, centroid.y), ha="center", fontsize=8, fontweight="bold")        ax.set_title("Ingresos Operacionales Promedio por Comuna - Cali 2025", fontsize=14, fontweight="bold")    ax.set_axis_off()    plt.tight_layout()        fig.savefig(OUTPUT_DIR / "mapa_ingresos_promedio_comuna.png", dpi=150, bbox_inches="tight")    print("[OK] Guardado: outputs/mapa_ingresos_promedio_comuna.png")    plt.show()else:    print("[!] No se puede generar mapa de ingresos.")

## 9. Mapa coropletico - Empleo total por comuna

In [ ]:
if gdf_merged is not None and gdf_merged["empleo_total"].notna().any():    fig, ax = plt.subplots(1, 1, figsize=(12, 10))        gdf_merged.plot(        column="empleo_total",        cmap="Greens",        linewidth=0.8,        edgecolor="0.3",        legend=True,        legend_kwds={"label": "Empleo total", "shrink": 0.7},        ax=ax,        missing_kwds={"color": "lightgrey", "label": "Sin datos"}    )        for idx, row in gdf_merged.iterrows():        centroid = row.geometry.centroid        label = str(int(row["comuna_num"])) if pd.notna(row.get("comuna_num")) else ""        ax.annotate(label, xy=(centroid.x, centroid.y), ha="center", fontsize=8, fontweight="bold")        ax.set_title("Empleo Total por Comuna - Cali 2025", fontsize=14, fontweight="bold")    ax.set_axis_off()    plt.tight_layout()        fig.savefig(OUTPUT_DIR / "mapa_empleo_total_comuna.png", dpi=150, bbox_inches="tight")    print("[OK] Guardado: outputs/mapa_empleo_total_comuna.png")    plt.show()else:    print("[!] No se puede generar mapa de empleo.")

## 10. Mapa interactivo con Folium

In [ ]:
import foliumif gdf_merged is not None and gdf_merged["total_empresas"].notna().any():    # Centro de Cali    centro = [3.4516, -76.5320]        m = folium.Map(location=centro, zoom_start=12, tiles="CartoDB positron")        # Capa coropletica - Total empresas    folium.Choropleth(        geo_data=gdf_merged.to_json(),        data=gdf_merged,        columns=["comuna_key", "total_empresas"],        key_on="feature.properties.comuna_key",        fill_color="YlOrRd",        fill_opacity=0.7,        line_opacity=0.5,        legend_name="Total de empresas por comuna",        name="Densidad empresarial"    ).add_to(m)        # Tooltips con info    style_function = lambda x: {"fillOpacity": 0, "weight": 0}        folium.GeoJson(        gdf_merged.to_json(),        name="Info por comuna",        tooltip=folium.GeoJsonTooltip(            fields=["comuna_key", "total_empresas", "empleo_total", "pct_micro", "n_ciiu_distintos", "densidad_empresarial"],            aliases=["Comuna", "Empresas", "Empleo total", "% Micro", "Actividades CIIU", "Empresas/ha"],            localize=True        ),        style_function=style_function    ).add_to(m)        # Capas base    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)    folium.TileLayer(        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",        attr="Esri",        name="Esri Satelite"    ).add_to(m)        folium.LayerControl().add_to(m)        # Guardar HTML    m.save(str(OUTPUT_DIR / "mapa_interactivo_comunas.html"))    print("[OK] Guardado: outputs/mapa_interactivo_comunas.html")        melse:    print("[!] No se puede generar mapa interactivo.")

## 11. Tabla resumen de indicadores por comuna

In [ ]:
if gdf_merged is not None:    resumen = gdf_merged[[        "comuna_key", "total_empresas", "ingresos_promedio", "ingresos_total",        "empleo_total", "pct_micro", "n_ciiu_distintos", "densidad_empresarial"    ]].copy()        resumen = resumen.sort_values("total_empresas", ascending=False)        # Formatear valores grandes    resumen["ingresos_prom_fmt"] = resumen["ingresos_promedio"].apply(        lambda x: f"${x/1e6:.1f}M" if pd.notna(x) else "N/A"    )    resumen["ingresos_total_fmt"] = resumen["ingresos_total"].apply(        lambda x: f"${x/1e9:.2f}B" if pd.notna(x) and x >= 1e9 else (f"${x/1e6:.0f}M" if pd.notna(x) else "N/A")    )        cols_display = ["comuna_key", "total_empresas", "ingresos_prom_fmt", "ingresos_total_fmt",                    "empleo_total", "pct_micro", "n_ciiu_distintos", "densidad_empresarial"]        print("INDICADORES POR COMUNA - Registro Mercantil Cali 2025")    print("=" * 90)    print(resumen[cols_display].to_string(index=False))else:    print("[!] No hay datos cruzados disponibles.")

## 12. Notas- **Entorno local:** Usar `uv run` o activar el ambiente conda. Dependencias: pandas, openpyxl, geopandas, folium, matplotlib- **Entorno Colab:** Descomentar la celda de instalacion de paquetes. El repo se clona automaticamente.- **GeoJSON:** Ubicado en `data/info_geo/geojson_comunas/Comunas.geojson` (22 comunas de Cali)- **Match de comunas:** El GeoJSON usa "Comuna 6" y el Registro Mercantil "Comuna 06". Se normaliza con zero-padding.- **Comunas faltantes:** El GeoJSON tiene 22 comunas, el Registro Mercantil tiene 39 zonas (incluye corregimientos). Solo se mapean las que coinciden.- **Salidas:** PNG en `outputs/`, HTML interactivo en `outputs/`